In [ ]:
from plotting_functions import cart2sph
from main_functions import *
from main_functions_generalAPI import *
from plotting_functions_generalAPI import *
import os, h5py, pickle
import numpy as np
from scipy.optimize import minimize
from scipy.interpolate import interp1d
import matplotlib
import matplotlib.pyplot as plt
%matplotlib qt

In [ ]:
plane=0
# 2026-02-25_mb_fish1_rec2
recording_name = '2026-03-04_mb_fish8_rec2'
#recording_name='2026-02-25_mb_fish1_rec2'
recording_path = os.path.join('data', recording_name)
save_path = os.path.join('results', recording_name, 'plane'+str(plane))

# set flags for saving intermediate results
compute_dff=True # False skip recalculating dff

# load eyetracking data
camera = h5py.File(os.path.join(recording_path, 'Camera.hdf5'), 'r')
fluorescence, rec, phase, ca_rec_group_id_fun = digest_folder(recording_path, imaging_rate=1.9957, plane=4)
# add resampled CMN data to resampled time domain for each phase
# (info: time domain is resampled from ~2.1Hz to 10Hz in digest_folder())
process_recording(rec, phase, radial_bin_num=16)
if  compute_dff:
    dff_original, dff_resampled = calculate_dff_vectorized(rec, fluorescence, rec['imaging_rate'])
    Path(save_path).mkdir(parents=True, exist_ok=True)
    np.save(os.path.join(save_path, "dff_original.npy"), dff_original)
else:
    dff_original = np.load(os.path.join(save_path, "dff_original.npy"))
    dff_resampled = scipy.interpolate.interp1d(rec['ca_times'], dff_original, kind='nearest')(
            rec['time_resampled'])

eye_pos=np.column_stack(
    (np.array(camera['eyepos_ang_le_pos_0']).squeeze(),
     np.array(camera['eyepos_ang_re_pos_0']).squeeze()))

eye_pos_resampled = interp1d(np.array(camera['fish_embedded_frame_time']).squeeze(), eye_pos.T, kind='nearest')(rec['time_resampled']).T

# define eye position quadrants

In [ ]:
def plot_eyepositions(
        eyepos,
        q1_min_left,
        q1_min_right,
        q1_width,
        q1_height,
        q3_max_left,
        q3_max_right,
        q3_widht,
        q3_height):
    """
        Scatterplot of eye positions to visualize quadrants for subselecting data.
    """
    plt.figure(figsize=(20,20))
    plt.scatter(eyepos[:,0], eyepos[:,1], s=1., alpha=0.6)
    plt.xlabel('left')
    plt.ylabel('right')

    # 1st quadrant (defined by lower boundaries)
    plt.hlines((q1_min_right, q1_min_right+q1_height),q1_min_left, q1_min_left+q1_width)
    plt.vlines((q1_min_left, q1_min_left+q1_width),  q1_min_right, q1_min_right+q1_height)

    # 3rd quadrant (define by upper boundaries)
    plt.hlines((q3_max_right, q3_max_right-q3_height), q3_max_left-q3_widht, q3_max_left)
    plt.vlines((q3_max_left, q3_max_left-q3_widht), q3_max_right-q3_height, q3_max_right)


In [ ]:
params=(10,-5, 6, -12) # parameters_fish1_rec2
params=(8,-8, 7, -9) # parameters_fish8_rec2

# 1st quadrant (defined by lower boundaries)
# 3rd quadrant (define by upper boundaries)
plot_eyepositions(eye_pos,
                  q1_min_left=params[0],
                  q1_min_right=params[1],
                  q1_width=25,
                  q1_height=30,
                  q3_max_left=params[2],
                  q3_max_right=params[3],
                  q3_widht=20,
                  q3_height=20)

In [ ]:
# Eye tracking
q1_mask, q3_mask=generate_eyepos_masks(
    camera['eyepos_ang_le_pos_0'],
    camera['eyepos_ang_re_pos_0'],
    camera['fish_embedded_frame_time'],
    rec['time_resampled'],
    q1_min_left = params[0],
    q1_min_right = params[1],
    q3_max_left = params[2],
    q3_max_right = params[3])

# estimate RF

use 2D projections to calculate RSS_angle
if theres time, do this while still n 3D coordinated exclude potential artifacts from the 2D projection

In [ ]:
#E = rec['preferred_vectors'] # RF estimation from ETAs

In [ ]:
i=0
dff_i = dff_resampled[i]

# load bootstrapped data
_path=os.path.join(save_path, 'bootstrapped RBEs',)
q1_rbe_bootstrapped=np.load(os.path.join(_path, f'neuron_{str(i)}_bsRBE_q1.npy'))
q3_rbe_bootstrapped=np.load(os.path.join(_path, f'neuron_{str(i)}_bsRBE_q1.npy'))

# calcium events
rec['signal_selection'], rec['signal_length'], rec['signal_proportion'], rec['signal_dff_mean']\
    =detect_events_with_derivative_generalAPI(
    rec['cmn_phase_selection'],
    dff_i,
    rec['sample_rate'])

In [ ]:
# ETAs
# 1. based on data from all eye positions
radial_bin_norms, radial_bin_etas = calculate_local_directions_generalAPI(
    rec['cmn_motion_vectors_2d'][rec['signal_selection'], :, :],
    rec['radial_bin_edges'])
# 2. based on data from one side/quadrant of eye positions
radial_bin_norms_q1, radial_bin_etas_q1 = calculate_local_directions_generalAPI(
    rec['cmn_motion_vectors_2d'][rec['signal_selection'] & q1_mask, :, :],
    rec['radial_bin_edges'])
# 3. based on data from other side/quadrant of eye positions
radial_bin_norms_q3, radial_bin_etas_q3 = calculate_local_directions_generalAPI(
    rec['cmn_motion_vectors_2d'][rec['signal_selection'] & q3_mask, :, :],
    rec['radial_bin_edges'])

In [ ]:
# estimate RFs
RF_est=calc_preferred_directions_generalAPI(
    radial_bin_etas,
    np.ones_like(radial_bin_etas) > 0,
    rec['radial_bin_centers'])
RF_est_q1=calc_preferred_directions_generalAPI(
    radial_bin_etas_q1,
    np.ones_like(radial_bin_etas_q1) > 0,
    rec['radial_bin_centers'])
RF_est_q3=calc_preferred_directions_generalAPI(
    radial_bin_etas_q3,
    np.ones_like(radial_bin_etas_q3) > 0,
    rec['radial_bin_centers'])

# fit optic flow field to estimated RF

note: scipy minimize, dont care about values, since only angles count.
define rotation axis: azimuth angle, elevation angle



## functions TOF and ROF

In [ ]:
def tof(angle_azimuth: float, angle_elevation: float, P: np.ndarray):
    """
        Translational optic flow field.
        Returns a translational optic flow field for a given translation axis.

        Parameters:
            angle_azimuth (float):
                angle in radians of the azimuth of the translation axis.
            angle_elevation (float):
                angle in radians of the elevation of the translation axis.
            P (np.array):
                3D positions to sample flow field at.

        Returns:
            F (np.array):
                Return values of optic flow field (3D vector) at each point
                given in positions.
            FoC, FoE (np.array):
                Focus of Expansion and Focus of Contraction of the optic flow field.
                Given as 3D unit vector.
    """
    from numpy import sin, cos
    axis, a, b = np.array([1,0,0]), angle_azimuth, angle_elevation
    # yaw and pitch rotation matrices
    yaw=np.array([[cos(a),-sin(a),0],[sin(a), cos(a), 0],[0,0,1]])
    pitch=np.array([[cos(b), 0, sin(b)],[0,1,0],[-sin(b), 0, cos(b)]])
    T=axis @ yaw @ pitch
    return T - np.dot(P, T)[:,None] * P

def rof(angle_azimuth: float, angle_elevation: float, P: np.ndarray):
    """
        Rotational optic flow field.
        Returns a rotation optic flow field for a given translation axis.

        Parameters:
            angle_azimuth (float):
                angle in radians of the azimuth of the rotation axis.
            angle_elevation (float):
                angle in radians of the elevation of the rotation axis.
            P (np.array):
                3D positions to sample flow field at.

        Returns:
            F (np.array):
                Return values of optic flow field (3D vector) at each point
                given in positions.
            FoC, FoE (np.array):
                Focus of Expansion and Focus of Contraction of the optic flow field.
                Given as 3D unit vector.
    """
    from numpy import sin, cos
    axis, a, b = np.array([1,0,0]), angle_azimuth, angle_elevation
    # yaw and pitch rotation matrices
    yaw=np.array([[cos(a),-sin(a),0],[sin(a), cos(a), 0],[0,0,1]])
    pitch=np.array([[cos(b), 0, sin(b)],[0,1,0],[-sin(b), 0, cos(b)]])
    return -np.cross(P, axis @ yaw @ pitch)

## functions RSSangle

In [ ]:
def RSSangle(F, E):
    """
        Calculate RSS (residual sum of squared) of angles between two vector fields.
        Used in this analysis to calculate RSS between estimated RF and optic flow field,
        as an error function to fit the latter to the former.
        Parameters:
            F (np.array):
                Optic flow field to fit.
            E (np.array):
                Estimated RF.
        Returns:
            RSS (float)
    """
    #print(E)
    coeffcients = np.sum(F*E, axis=1)/ (np.linalg.norm(np.clip(E, 0.0000001, 1.), axis=1)* np.linalg.norm(F, axis=1))
    angles = np.arccos(np.clip(coeffcients, -1.0, 1.0))
    return np.sum(angles**2)

def RSSangle_Fto2D(F, E, pos):
    """
        Wraps around def RSSangle(F, E) to transform F from 3D to 2D,
        for convenience in using it with scipy.optimize.minimize
    """
    F=project_to_local_2d_vectors(pos, F[None,:,:]).squeeze()
    return RSSangle(F, E)


## executions

In [ ]:
radial_bin_centers, positions=rec['radial_bin_centers'], rec['positions']
radial_bin_centers.shape, positions.shape

In [ ]:
fig=plt.figure()
ax=fig.add_subplot(projection='3d')
ax.scatter(rec['positions'][:,0],rec['positions'][:,1],rec['positions'][:,2])
ax.scatter(0,0,0)
plt.show()

In [ ]:
TOF=tof(0, np.pi/4, positions)
positions=rec['positions']
TOF.shape, positions.shape

In [ ]:
# plot in 3D
ax = plt.figure().add_subplot(projection='3d')
ax.quiver(positions[:,0], positions[:,1], positions[:,2], TOF[:,0], TOF[:,1], TOF[:,2], length=0.2)
ax.set_zlabel('elevation[deg]')

In [ ]:
# mock data
# F = project_to_local_2d_vectors(positions, tof(1.,0., positions)[None,:,:]).squeeze()
#np.sum(F*E, axis=1)/ (np.linalg.norm(E, axis=1) * np.linalg.norm(F, axis=1))

In [ ]:
#E = rec['preferred_vectors'] # RF estimation from ETAs
E = RF_est_q1

In [ ]:
mini=minimize(
    lambda angles: RSSangle_Fto2D(tof(*angles, positions), E, rec['positions']),
    [0., 0.],
)

# calculate fitted optic flow field F from optimal parameters
F_3D=tof(*mini.x, positions)[None,:,:]
F=project_to_local_2d_vectors(positions, F_3D).squeeze()

In [ ]:
# Plot estimated RF and fitted TOF/ROF together
gs_cmn = plt.GridSpec(20, 10)
fig_cmn = plt.figure(figsize=(20, 10))
# Plot preferred local motion vectors quiver plot
ax_quiv = fig_cmn.add_subplot(gs_cmn[:, :])
# Get position and local motion preference data
x, y, _ = cart2sph(*positions.T)

Evels=np.linalg.norm(E, axis=1)
color = matplotlib.colormaps['tab20'](0)
ax_quiv.quiver(x, y, E[:, 0], E[:, 1],
               pivot='mid', color=color, width=0.002, scale=Evels.max() * 30)

Fvels=np.linalg.norm(F, axis=1)
color = matplotlib.colormaps['tab20'](2)
ax_quiv.quiver(x, y, F[:, 0], F[:, 1],
               pivot='mid', color=color, width=0.002, scale=Fvels.max() * 30)

ax_quiv.set_xlabel('azimuth [deg]')
ax_quiv.set_ylabel('elevation [deg]')
ax_quiv.set_aspect('equal')

In [ ]:
"""
    theoretisch zu tun
1. find translation sensitive neurons
2. calculate delta for all translation sensitive neurons
3. run a statistical test for the delta value
4. create visualizations for deltas and poster
5. show that difference could be shown, if there is one
"""

In [ ]:
"""
    poster
Grafiken
    Mikroskop setup, CMN stimulus -> Even triggered averaging
    Wie sieht ein global motion sensitive RF aus
    Was ist das delta ? Berechnet sich aus Focus of Contraction
    Wie sehen die Ergebnisse aus ?
        Signifikanztest fuer Verteilungen der beiden FoC fuer linke/rechte Augenposition
            oder ? correlation des FoC (bzw) der angle horizontal mit der Augenposition ?
        Histogramm der Verteilung der delta Werte

    """

In [ ]:
"""
    Histogramm der Differenzen der Verteilung der Augenpositionen im Vergleich zu dem Histogramm
    der Differenzen der
"""

# loop for all neurons

In [ ]:
n_neurons=dff_resampled.shape[0]
E_list_q1, E_list_q3 = [], []
F_list_q1, F_list_q3 = [], []
FoC_list_q1, FoC_list_q3 = [], []
FE1_sim_list, FE3_sim_list=[],[]


for i in tqdm(range(n_neurons)):
    # load bootstrapped etas
    _path=os.path.join(save_path, 'bootstrapped RBEs',)
    q1_rbe_bootstrapped=np.load(os.path.join(_path, f'neuron_{str(i)}_bsRBE_q1.npy'))
    q3_rbe_bootstrapped=np.load(os.path.join(_path, f'neuron_{str(i)}_bsRBE_q3.npy'))

    # calcium events
    rec['signal_selection'], rec['signal_length'], rec['signal_proportion'], rec['signal_dff_mean']\
        =detect_events_with_derivative_generalAPI(
        rec['cmn_phase_selection'],
        dff_resampled[i],
        rec['sample_rate'])

    # ETAs
    # 1. based on data from all eye positions
    radial_bin_norms_q1, radial_bin_etas_q1 = calculate_local_directions_generalAPI(
        rec['cmn_motion_vectors_2d'][rec['signal_selection'] & q1_mask, :, :],
        rec['radial_bin_edges'])
    # 3. based on data from other side/quadrant of eye positions
    radial_bin_norms_q3, radial_bin_etas_q3 = calculate_local_directions_generalAPI(
        rec['cmn_motion_vectors_2d'][rec['signal_selection'] & q3_mask, :, :],
        rec['radial_bin_edges'])

    # bin signficances based on bootstrapped etas
    significances_q1, pvalues_q1 = calculate_directional_significance_generalAPI(
        radial_bin_etas_q1,
        q1_rbe_bootstrapped)
    significances_q3, pvalues_q3 = calculate_directional_significance_generalAPI(
        radial_bin_etas_q3,
        q3_rbe_bootstrapped)

    # bin signficances for bootstrapped data, based on bootstrapped etas
    significances_bs_q1, pvalues_bs_q1 = calculate_directional_significance_permutations_generalAPI(
        q1_rbe_bootstrapped)
    significances_bs_q3, pvalues_bs_q3 = calculate_directional_significance_permutations_generalAPI(
        q3_rbe_bootstrapped)

    # estimate RFs (cluster statistics are not used in this)
    RF_est_q1=calc_preferred_directions_generalAPI(
        radial_bin_etas_q1,
        significances_q1 > 0,
        rec['radial_bin_centers'])
    RF_est_q3=calc_preferred_directions_generalAPI(
        radial_bin_etas_q3,
        significances_q3 > 0,
        rec['radial_bin_centers'])

    # find clusters
    full_indices_q1, unique_indices_q1, bs_cluster_full_indices_q1, bs_cluster_unique_indices_q1=find_clusters_generalAPI(
        significances_q1,
        significances_bs_q1,
        rec['closest_3_position_indices'],)
    full_indices_q3, unique_indices_q3, bs_cluster_full_indices_q3, bs_cluster_unique_indices_q3=find_clusters_generalAPI(
    significances_q3,
    significances_bs_q3,
    rec['closest_3_position_indices'],)

    # calculate cluster statistics
    original_cluster_scores_q1, bs_max_cluster_scores_q1, cluster_significant_indices_q1 = calculate_cluster_significances_generalAPI(
        full_indices_q1,
        bs_cluster_full_indices_q1,
        significances_q1,
        significances_bs_q1)
        # calculate cluster statistics
    original_cluster_scores_q3, bs_max_cluster_scores_q3, cluster_significant_indices_q3 = calculate_cluster_significances_generalAPI(
        full_indices_q3,
        bs_cluster_full_indices_q3,
        significances_q3,
        significances_bs_q3)

    # plot

    # fit E (translational/ rotational optic flow field)
    E1 = RF_est_q1
    mini=minimize(
    lambda angles: RSSangle_Fto2D(tof(*angles, rec['positions']), E1, rec['positions']),
    [0., 0.],
    )
    # calculate fitted optic flow field F from optimal parameters
    F1_3D=tof(*mini.x, rec['positions'])[None,:,:]
    F1=project_to_local_2d_vectors(rec['positions'], F1_3D).squeeze()
    E_list_q1.append(E1)
    F_list_q1.append(F1)
    FoC_list_q1.append(mini.x)

    E3 = RF_est_q3
    mini=minimize(
    lambda angles: RSSangle_Fto2D(tof(*angles, rec['positions']), E3, rec['positions']),
    [0., 0.],
    )
    # calculate fitted optic flow field F from optimal parameters
    F3_3D=tof(*mini.x, rec['positions'])[None,:,:]
    F3=project_to_local_2d_vectors(rec['positions'], F3_3D).squeeze()
    E_list_q3.append(E3)
    F_list_q3.append(F3)
    FoC_list_q3.append(mini.x)

    FE1_sim_list.append(FE_similarity(F1, E1))
    FE3_sim_list.append(FE_similarity(F3, E3))

    plot_v1(
        E1, E3,
        F1, F3,
        rec['positions'],
        alpha_E=.9, alpha_F=.55,
        save_path_=save_path,
        neuron_num=i)


FoC_array_q1=np.concatenate([FoC[:,None].T for FoC in FoC_list_q1])
FoC_array_q3=np.concatenate([FoC[:,None].T for FoC in FoC_list_q3])

In [ ]:
FoC_array_q1[np.abs(FoC_array_q1) > 3.] = np.nan
FoC_array_q3[np.abs(FoC_array_q3) > 3.] = np.nan
# np.isnan(FoC_array_q1).sum(), np.isnan(FoC_array_q3).sum()

In [ ]:
plt.hist(
    (eye_pos_resampled[q1_mask,0],
     eye_pos_resampled[q3_mask, 0]),
    bins=140,)
plt.hist(
    (FoC_array_q1[:,0], FoC_array_q3[:,0]),
    bins=140,)

In [ ]:
scipy.stats.wilcoxon(FoC_array_q1[:,0], FoC_array_q3[:,0], nan_policy='omit')

# plot estimated receptive fields

In [ ]:
def plot_v1(E1, E3, F1, F3, positions, save_path_=None, neuron_num=None, alpha_E=.5, alpha_F=.5):
    # Plot estimated RF and fitted TOF/ROF together
    fig, (ax1, ax3)=plt.subplots(ncols=1, nrows=2, figsize=(20, 10))
    # Plot preferred local motion vectors quiver plot
    # Get position and local motion preference data
    x, y, _ = cart2sph(*positions.T)

    E1vels=np.linalg.norm(E1, axis=1)
    color = matplotlib.colormaps['tab20'](0)
    ax1.quiver(
        x, y,
        E1[:, 0], E1[:, 1],
        pivot='mid',color=color,width=0.002,scale=E1vels.max() * 30,alpha=alpha_E)

    F1vels=np.linalg.norm(F1, axis=1)
    color = matplotlib.colormaps['tab20'](2)
    ax1.quiver(
        x, y,
        F1[:, 0], F1[:, 1],
        pivot='mid',color=color,width=0.002,scale=F1vels.max() * 30,alpha=alpha_F)

    E3vels=np.linalg.norm(E3, axis=1)
    color = matplotlib.colormaps['tab20'](0)
    ax3.quiver(
        x, y,
        E3[:, 0], E3[:, 1],
        pivot='mid',color=color,width=0.002,scale=E3vels.max() * 30,alpha=alpha_E)

    F3vels=np.linalg.norm(F3, axis=1)
    color = matplotlib.colormaps['tab20'](2)
    ax3.quiver(
        x, y,
        F3[:, 0], F3[:, 1],
        pivot='mid',color=color,width=0.002,scale=F3vels.max() * 30,alpha=alpha_F)

    ax1.set_xlabel('azimuth [deg]')
    ax3.set_xlabel('azimuth [deg]')
    ax1.set_ylabel('elevation [deg]')
    ax3.set_ylabel('elevation [deg]')
    ax1.set_aspect('equal')
    ax3.set_aspect('equal')

    if save_path_ is not None and neuron_num is not None:
        png_path = os.path.join(save_path_, 'png')
        pdf_path = os.path.join(save_path_, 'pdf')
        Path(png_path).mkdir(parents=True, exist_ok=True)
        Path(pdf_path).mkdir(parents=True, exist_ok=True)
        fig_cmn.savefig(f'{png_path}/{neuron_num}.png', dpi=300)
        fig_cmn.savefig(f'{pdf_path}/{neuron_num}.pdf', dpi=300)
        plt.close(fig_cmn)


In [ ]:
plot_v1(E1, E3, F1, F3, rec['positions'], alpha_E=.9, alpha_F=.55)

## calculate similarity between fitted optic flow field and estimated RF

In [ ]:
def FE_similarity(F, E):
    return ((np.sum(F*E, axis=1)/
            (np.linalg.norm(np.clip(E, 0.0000001, 1.), axis=1) *
             np.linalg.norm(F, axis=1)))
            .mean())

In [ ]:
FE_similarity(F1, E1)

# subselect neurons

## 8% threshold for neural activity

In [ ]:
proportions=[]
for dff_i in dff_resampled:
    signal_selection, signal_length, signal_proportion, signal_dff_mean\
        =detect_events_with_derivative_generalAPI(
        rec['cmn_phase_selection'],
        dff_i,
        rec['sample_rate'])
    proportions.append(signal_proportion)

In [ ]:
prop=np.array(proportions)

In [ ]:
plt.hist(prop, bins=80)

In [ ]:
phase_visual = [[name, phase[name]['__visual_name']] for name in phase.keys() if name.startswith('phase')]
phase_visual = np.array(phase_visual)
phase_visual

## based on response intensitites > 3

In [ ]:
phase_id=99

In [ ]:
dff_original[:,phase['in_phase_idcs_'+str(phase_id)]].max(axis=1).shape

In [ ]:
(dff_original[:,phase['in_phase_idcs_'+str(phase_id)]].max(axis=1)-
 dff_original[:,phase['in_phase_idcs_'+str(phase_id)][0]]).shape

In [ ]:
R=np.stack([np.maximum(0,
                       dff_original[:,phase['in_phase_idcs_'+str(phase_id)]].max(axis=1)-
                       dff_original[:,phase['in_phase_idcs_'+str(phase_id)][0]]
                       )
            for phase_id in range(105)], axis=1)
R.shape

In [ ]:
strong_response_indices_mask = R.max(axis=0)>3
strong_response_indices_mask.sum()

In [ ]:
plt.hist(R.flatten(), bins=2000)

## exlude with drift or whitening from first to last 16 seconds

In [ ]:
t = rec['time_resampled']
t0 = t[0]

In [ ]:
# masks for fist and last 16 seconds
start, end= (t-t[0] < 16), (t > t[-1] - 16.)

In [ ]:
mu_end, mu_start=dff_resampled[:,end].mean(axis=1), dff_resampled[:,start].mean(axis=1)
std_end, std_start=dff_resampled[:,end].std(axis=1), dff_resampled[:,start].std(axis=1)

In [ ]:
np.sum(np.abs(mu_start-mu_end)>4*mu_start), sum((std_end/std_start) > 4)

# cluster functions general_API

good for plotting receptive fields of eyepositions, probably good for poster ?

In [ ]:
# full_indices, unique_indices, bs_cluster_full_indices, bs_cluster_unique_indices=find_clusters_generalAPI(
#     q1_significances,
#     q1_bs_significances,
#     rec['closest_3_position_indices'],
# )

In [ ]:
# original_cluster_scores, bs_max_cluster_scores, cluster_significant_indices = calculate_cluster_significances_generalAPI(
#     full_indices,
#     bs_cluster_full_indices,
#     q1_significances,
#     q1_bs_significances)

# inspect coordinate system

In [ ]:
normals=rec['positions']
vectors=rec['cmn_motion_vectors_3d']
normals.shape, vectors.shape

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(projection='3d')

# x-axis: left-right, y=axis: fore-back, z=axis: up-down
ax.scatter(rec['positions'][:, 0], rec['positions'][:, 1], rec['positions'][:, 2])

#ax.scatter(0, 0, 0)
# assume the fishs FOV is centered here
ax.scatter(1, 0, 0)

plt.show()

In [ ]:
# vectical norms
vnorms = np.array([0, 0, 1]) - normals * np.dot(normals, np.array([0, 0, 1]))[:, None]
vnorms.shape

In [ ]:
np.dot(normals, np.array([0, 0, 1]))[:, None].shape

In [ ]:
vnorms /= np.linalg.norm(vnorms, axis=1)[:, None]

# horizontal norms
hnorms = -crossproduct(vnorms, normals)
hnorms /= np.linalg.norm(hnorms, axis=1)[:, None]

vectors_2d = np.zeros((*vectors.shape[:2], 2))
for i, v in enumerate(vectors):
    # Calculate 2d motion vectors in coordinate system defined by local horizontal and vertical norms
    motvecs_2d = np.array([np.sum(v * hnorms, axis=1),
                           np.sum(v * vnorms, axis=1)])
    vectors_2d[i] = motvecs_2d.T